# 01 — Data and features

Two surveys, asked in different countries with different instruments, have to end up describing
the same thirty-one things about an adolescent and the same five kinds of adverse experience
before any model can be carried from one to the other. That reduction is what this notebook does,
and the decisions it takes are the ones every later number rests on.

It stops at the pillars. The outcome is a deterministic collapse of them through `make_outcome`
and the splits are deterministic given features, outcome and seed, so notebook 02 composes both
from what is persisted here and neither is stored twice.

**Governance.** YRBS 2023 is open CDC data. MCS Sweep 6 is UKDS Safeguarded **Tier 1a** and
resolves through `$MCS_DATA_DIR`, outside this repository.

No raw or person-level respondent row is printed or displayed here — every diagnostic on screen
is an aggregate. Person-level MCS frames **are** persisted: section F writes three of them, and
they go only to the configured restricted MCS location, never into the repository. The optional
exact-count file in section E is written to that same location. Section D sets out how small
counts are reported.

## A — Inputs and access

Six inputs. The distinction that matters most is the middle column: two are survey data that must
be put in place by hand, four are tracked definitions that travel with the repository.

In [4]:
# Reading MCS requires the acknowledgement to be set in the environment this notebook is
# launched from. It is deliberately not set here: a tracked file that opens the gate opens it
# for anyone who runs the file.
import os

if not os.environ.get("MCS_READ_OK"):
    raise RuntimeError(
        "MCS is UKDS Safeguarded Tier 1a. Launch this notebook from a shell where the "
        "acknowledgement is set, then re-run. Nothing below reads MCS until it is.")

In [5]:
import sys

# src/ is a sibling of notebooks/; only one of these two exists from any
# working directory, and a path that does not exist is ignored.
sys.path[:0] = ["src", "../src"]

import pandas as pd

import config
import data
import features
import inputs
import outcomes
import publication

# Presentation mode, and nothing else. False is the internal notebook and is what a run
# executes: every diagnostic is shown. scripts/make_public_notebooks.py writes a SEPARATE
# copy with this set to True and keeps output only in the cells
# spec/public_notebook_cells.json names. Both settings compute, validate and write exactly
# the same files; only what is displayed differs. Neither clears anything for release.
PUBLIC_NOTEBOOK = True

**Public output.** This is the internal notebook: it declares `PUBLIC_NOTEBOOK = False` and
shows every diagnostic. It computes the five tables the manuscript needs — cohort flow,
sample characteristics, outcome prevalence, missingness and the YRBS inclusion comparison —
and writes them as review candidates below the configured working root, in either mode. A public copy, made by
`scripts/make_public_notebooks.py`, is a separate file that displays the approved subset.

Some MCS-derived values are not reported following disclosure-control review. Additional values may be withheld to prevent recovery by subtraction or comparison across tables. A withheld value appears as `—` wherever it would otherwise be shown, and the
same marker is used whatever the reason, so the marker itself says nothing about the value. The
internal frames are still computed; they are simply not shown.

**None of that is disclosure approval.** Every MCS-derived figure here needs my review, and the
release rules of the UK Data Service agreement, before it leaves the approved environment.

Six inputs. Only two are files that must be supplied by hand; the rest are tracked here.

In [6]:
print(f"{len(features.FEATURE_MAP)} model features, "
      f"{len(features.ATTRIBUTE_MAP)} evaluation-only attributes, "
      f"{len(outcomes.SHARED_PILLARS)} shared ACE pillars "
      f"({len(outcomes.MCS_PILLARS)} MCS, {len(outcomes.YRBS_PILLARS)} YRBS)")

30 model features, 4 evaluation-only attributes, 5 shared ACE pillars (5 MCS, 8 YRBS)


### Supplying the two survey files

**Neither cohort is downloaded.** `data.download_yrbs_2023` returns the YRBS parquet's path when
it is in place and otherwise raises with the CDC instructions — there is no downloader here. For
YRBS that is a missing convenience; MCS cannot be automated at all, needing an accepted UKDS
application for SN 8156. `README.md` records both routes.

In [7]:
yrbs_raw_path = data.download_yrbs_2023(config.YRBS_RAW.parent, force=False)
print(f"YRBS input in place: {yrbs_raw_path.name} "
      f"({yrbs_raw_path.stat().st_size / 1e6:.1f} MB)")

### Reading the two cohorts

MCS follows children born in 2000–2002, interviewed at age 14; YRBS respondents are 14 to 18.
The age gap is a limitation the paper states rather than something to correct — both surveys ask
about lifetime exposure, so a wider age range means more exposure time, not a different construct.

**Where this stops without MCS access.** `config` needs no credentials to import; it resolves a
path on attribute access. So the first MCS name is reached inside `data.load_raw("mcs")` in the
cell below, and that is where an unset `$MCS_DATA_DIR` raises. A locally-installed read gate, if
one is present, intercepts at the same point with its own message.

In [8]:
mcs_raw = data.load_raw("mcs")
yrbs_raw = data.load_raw("yrbs")

# Every frame built below inherits this index, so a repeated label would silently break the
# alignment between a respondent's features, pillars and attributes.
for label, raw in (("MCS", mcs_raw), ("YRBS", yrbs_raw)):
    if not len(raw):
        raise ValueError(f"{label}: the raw frame is empty")
    if not raw.index.is_unique:
        raise ValueError(f"{label}: the raw frame has repeated index labels, so features, "
                         f"pillars and attributes could not be aligned")

print(f"MCS {mcs_raw.shape[0]:,} respondents x {mcs_raw.shape[1]} variables")
print(f"YRBS {yrbs_raw.shape[0]:,} respondents x {yrbs_raw.shape[1]} variables")

The shapes above are what **this extract produced** — the two `raw_N` figures in the
manuscript's Table I. That reconciliation statement holds for every count and prevalence in this
notebook: **a difference from the manuscript value means the current extract or processing
differs and requires reconciliation; it is not automatically repaired or forced to match.**

---
## B — Harmonisation decisions

Both cohorts are whole surveys at this point. What can be carried between them is only the part
they genuinely share, and deciding what that is came before any of the construction below.

`spec/harmonisation_spec_v5.csv` is **a tracked project decision record**: one row per candidate
construct, with each cohort's variable and scale, the recode rule, a rationale and a keep-or-drop
decision. **The cohort-construction functions do not consume it**: the cell below reads it so
this section can show the decisions on record. The registries — `features.FEATURE_MAP`,
`outcomes.MCS_PILLARS` / `YRBS_PILLARS`, `features.ATTRIBUTE_MAP` — are the authority the
construction in section C uses. It is required for this section and not for building a cohort.

The counts below say which options are currently recorded under each decision. They do not say
when any decision was taken, and nothing reconciles the record against the registries: the record
keys a construct by prose and source variable, the registries by harmonised column name, and no
shared key exists to join them on.

In [9]:
spec = pd.read_csv(config.PROJECT_ROOT / "spec" / "harmonisation_spec_v5.csv")
print(f"{len(spec)} candidate constructs screened; currently recorded as:")
for decision, n in spec["keep_drop"].str.strip().value_counts().items():
    print(f"   {decision:<20} {n}")

parked = spec[spec["keep_drop"].str.strip() == "needs_review"]
for _, row in parked.iterrows():
    print(f"\nparked, not adopted: {row['construct']!r} ({row['domain']})")
    print(f"   {row['rationale']}")

86 candidate constructs screened; currently recorded as:
   keep                 35
   yrbs_only            30
   drop                 11
   mcs_only             5
   attribute_fairness   4
   needs_review         1

parked, not adopted: 'SMFQ continuous score' (Feature: Mental health)
   MISMATCH: continuous MCS vs binary YRBS. Cannot transfer continuous score cleanly


### Three decisions the rest of the notebook rests on

**The shared outcome is five pillars, not eight.** Neglect and parental separation are asked only
in YRBS; witnessed domestic violence is asked in both but could not be reconciled. None of the
three can enter an outcome that has to mean the same thing on both sides, so the shared outcome
is the five that survive.

**Four MCS scales run backwards.** `FCPHEX00` (physical activity), `FCSWTD00` (soda),
`FCHURT00` (bullied) and `FCCYBU00` (cyberbullied) are coded so a higher raw value means *less*
of the thing, and all four are flipped to match their YRBS counterparts. An inversion would
corrupt every transfer number without changing a single row count, and cyberbullying is reported
in the manuscript as the highest-importance feature. Each flip is stated in the docstring of the
recoder that performs it.

**Depression was taken as binary, and the continuous alternative was parked.** MCS carries the
13-item SMFQ as a 0–26 score; YRBS has only the binary Q26. The two were not considered directly
transferable, so the adopted feature is the settled binary construct — SMFQ thresholded to 0/1.
The continuous alternative stays in the record marked `needs_review` and is not the feature the
pipeline uses.

This explains the analytical role of these decisions; it does not establish when they were made.

### What each harmonised column is built from

The record is keyed by construct and the frames by harmonised column. This table is the join
between them and the only place it is written down. **It is a manually maintained decision record
and should be reviewed when the recoding logic changes** — nothing derives it from the recoders,
so it does not update itself. The authoritative set is `features.FEATURE_MAP`; the check cell
below reconciles this table against it rather than trusting either alone.

| harmonised column | MCS variable(s) | YRBS question(s) |
|---|---|---|
| **model features** | | |
| `h_ever_smoked` | FCSMOK00 | Q31 |
| `h_current_cigarette` | FCSMOK00 | Q33 |
| `h_age_first_smoked` | FCAGSM00 | Q1, Q32 |
| `h_ecig_ever` | FCECIG00 | Q35 |
| `h_ecig_current` | FCECIG00 | Q36 |
| `h_ever_drank` | FCALCD00 | Q41 |
| `h_age_first_drank` | FCALAG00 | Q1, Q41 |
| `h_past_month_alcohol` | FCALNF00 | Q42 |
| `h_ever_binge` | FCALFV00 | Q43 |
| `h_past_year_binge_freq` | FCALFN00 | Q43 |
| `h_cannabis_freq` | FCCANB00, FCCANO00 | Q46 |
| `h_other_drugs` | FCOTDR00 | Q49, Q50, Q51, Q52, Q53, Q54, Q55 |
| `h_weapon_carrying` | FCKNIF00 | Q12, Q13 |
| `h_weapon_victim` | FCVICC00 | Q15 |
| `h_physical_fight` | FCHITT00, FCWEPN00 | Q16 |
| `h_bullied_school` | FCHURT00 | Q24 |
| `h_cyberbullied` | FCCYBU00 | Q25 |
| `h_depression` | FCMDSA00, FCMDSB00, FCMDSC00, FCMDSD00, FCMDSE00, FCMDSF00, FCMDSG00, FCMDSH00, FCMDSI00, FCMDSJ00, FCMDSK00, FCMDSL00, FCMDSM00 | Q26 |
| `h_self_harm` | FCHARM00 | Q29 |
| `h_recent_sex_condom_status` | FCSEXX00, FCCONP0A | Q59, Q61 |
| `h_breakfast_days` | FCBRKN00 | Q75 |
| `h_sugary_drinks` | FCSWTD00 | Q74 |
| `h_fruit_intake` | FCFRUT00 | Q69 |
| `h_veg_intake` | FCVEGI00 | Q70, Q71, Q72, Q73 |
| `h_mvpa_days` | FCPHEX00 | Q76 |
| `h_sleep_duration` | FCSLWK00, FCWUWK00 | Q85 |
| `h_weight_perception` | FCWEGT00 | Q66 |
| `h_social_media` | FCSOME00 | Q80 |
| `h_dentist_12mo` | FCDENY00 | Q83 |
| `h_weight_status_iotf` | FCOBFLG6 | Q1, Q2, Q6, Q7 |
| **5 outcome pillars** | | |
| `ace_sexual_abuse` | FCVICF0A | Q88 |
| `ace_emotional_abuse` | FCVICG00 | Q89 |
| `ace_physical_abuse` | FCVICA00 | Q90 |
| `ace_household_substance` | FPALDR00, FPDRUG00 | Q100 |
| `ace_household_mental_illness` | FPDEAN00 | Q101 |
| **5 evaluation-only attributes** | | |
| `attr_sex` | FCCSEX00 | Q2 |
| `attr_age` | FCMCS6AG | Q1 |
| `attr_ethnicity` | FDCE0600 | Q5, raceeth |
| `attr_orientation` | FCCSEX00, FCROMB00, FCROMG00 | Q64 |
| `attr_ethnicity_coarse` | FDCE0600 | raceeth |

Where a column draws on more than one variable the recode combines them — `attr_orientation`
needs three MCS items against a single YRBS question, and `h_age_first_smoked` needs YRBS age
(`Q1`) as well as the age-first question, because the target scale is years before 14.

**One column is nominal, and it is the only one.** `h_recent_sex_condom_status` combines each
cohort's recent-intercourse item with its condom item into three states —
`no_recent_intercourse`, `recent_no_condom`, `recent_condom` — because neither construct
transfers on its own: MCS asks about condom use only of respondents who reported recent
intercourse, so a condom answer is not interpretable without the intercourse answer beside it.
The codes are level names rather than a scale, so `data.model_features` enters it as two
indicators against `no_recent_intercourse` and no model ever sees the 0/1/2 column.

**MCS distinguished intercourse in the previous 12 months, while YRBS `Q59` used the previous
three months. MCS availability additionally depended on self-completion and preceding contact
routing. Only explicit, internally consistent states were coded; administrative, non-response
and indeterminate cases remained missing.**

**What the record cannot settle.** `recon_type` holds values 1–8 with no legend in the
repository; there is no conditionality flag, so "four are asked conditionally in MCS" is
unverifiable here; and `target_scale` is free text with four spellings of "binary", so the
16/12/2/1 type split cannot be recomputed. The first two remain unresolved: they are recorded here as limitations on what
this record can establish, not as settled questions.

---
## C — Cohort construction

Both cohorts pass through the **same** registries and the same builders. That is the point of the
step rather than an implementation convenience: it is what makes "the same thirty-one features"
a fact about the code rather than a claim about two parallel scripts.

A recoder that fails stops the build. A frame quietly short of a predictor would train a
different model from the one the manuscript reports, and nothing downstream would notice.

In [10]:
mcs_features = data.build_harmonised_features(mcs_raw, "mcs")
yrbs_features = data.build_harmonised_features(yrbs_raw, "yrbs")
mcs_pillars = data.build_pillars(mcs_raw, "mcs")
yrbs_pillars = data.build_pillars(yrbs_raw, "yrbs")
mcs_attributes = data.build_attributes(mcs_raw, "mcs")
yrbs_attributes = data.build_attributes(yrbs_raw, "yrbs")

for label, raw, feats, pill, attrs in (
        ("MCS", mcs_raw, mcs_features, mcs_pillars, mcs_attributes),
        ("YRBS", yrbs_raw, yrbs_features, yrbs_pillars, yrbs_attributes)):
    print(f"{label:<5} {raw.shape[1]:>4} variables -> {feats.shape[1]} features, "
          f"{pill.shape[1]} pillars, {attrs.shape[1]} attributes")

### What the construction has to have got right

In [11]:
FEATURES = list(features.FEATURE_MAP)
ATTRIBUTES = list(features.ATTRIBUTE_MAP)

# The registries are the authority: same columns, declared order. All three frames inherit the
# raw index, so they must still agree on it — that inheritance is the only thing holding a
# respondent's features, pillars and attributes together.
for label, feats, pill, attrs, reg in (
        ("MCS", mcs_features, mcs_pillars, mcs_attributes, outcomes.MCS_PILLARS),
        ("YRBS", yrbs_features, yrbs_pillars, yrbs_attributes, outcomes.YRBS_PILLARS)):
    if not len(feats):
        raise ValueError(f"{label}: the feature frame is empty")
    if list(feats.columns) != FEATURES:
        raise ValueError(f"{label}: features differ from FEATURE_MAP")
    if list(pill.columns) != list(reg):
        raise ValueError(f"{label}: pillars differ from the registry")
    # Section D adds `attr_ethnicity_coarse` to these same frames, so that one column is
    # tolerated and nothing else is — which is what lets this cell be re-run afterwards.
    if list(attrs.columns)[:len(ATTRIBUTES)] != ATTRIBUTES:
        raise ValueError(f"{label}: the base attributes differ from ATTRIBUTE_MAP")
    unexpected = sorted(set(attrs.columns) - set(ATTRIBUTES) - {"attr_ethnicity_coarse"})
    if unexpected:
        raise ValueError(f"{label}: unexpected attribute column(s) {unexpected}")
    if not feats.index.is_unique:
        raise ValueError(f"{label}: repeated index labels in the feature frame")
    if not feats.index.equals(pill.index) or not feats.index.equals(attrs.index):
        raise ValueError(f"{label}: features, pillars and attributes are not aligned")

if list(mcs_features.columns) != list(yrbs_features.columns):
    raise ValueError("the two cohorts do not share a feature schema")
if set(data.SHARED_PILLARS) != set(mcs_pillars.columns) & set(yrbs_pillars.columns):
    raise ValueError("the shared pillars are not the five SHARED_PILLARS declares")

print(f"{len(FEATURES)} features, identical and in registry order across both cohorts")
print(f"{len(data.SHARED_PILLARS)} shared pillars; YRBS carries "
      f"{yrbs_pillars.shape[1] - len(data.SHARED_PILLARS)} more that MCS cannot measure")

**A limitation of these checks.** Alignment is guaranteed through inherited index labels.
Notebook 01 does not independently verify a respondent identifier from the raw cohorts — no
identifier column is set as the index anywhere in the construction path, so what is established
above is that the three frames agree on the index they inherited, not that the index identifies
a respondent.

### The analytic sample

A respondent enters the analysis only if **all five shared pillars** are observed. That is the
cross-cohort rule, and it is the five and not the eight: YRBS measures three further pillars, but
a respondent is not excluded from this analysis for missing one of them, because those three
cannot enter an outcome that has to mean the same thing on both sides.

`analytic_sample_mask` reaches the same rule internally, through `make_outcome`, but the five are
subset explicitly below so the choice is visible where it applies. The mask is derived from
pillars rather than from a composed outcome because once composed the outcome cannot tell
"pillar missing" from "pillar observed and zero". Every later section uses this mask.

In [12]:
# Subset to the five shared pillars, so the inclusion rule is visible rather than implied.
mcs_shared_pillars = mcs_pillars[list(data.SHARED_PILLARS)]
yrbs_shared_pillars = yrbs_pillars[list(data.SHARED_PILLARS)]

mcs_analytic = data.analytic_sample_mask(mcs_features, outcome=None, side="mcs",
                                         pillars=mcs_shared_pillars)
yrbs_analytic = data.analytic_sample_mask(yrbs_features, outcome=None, side="yrbs",
                                          pillars=yrbs_shared_pillars)

for label, mask in (("MCS", mcs_analytic), ("YRBS", yrbs_analytic)):
    if not int(mask.sum()):
        raise ValueError(f"{label}: no respondent has all {len(data.SHARED_PILLARS)} shared "
                         f"pillars observed, so every prevalence below would describe nothing")

print(f"inclusion requires all {len(data.SHARED_PILLARS)} shared pillars: "
      f"{list(data.SHARED_PILLARS)}")

inclusion requires all 5 shared pillars: ['ace_sexual_abuse', 'ace_emotional_abuse', 'ace_physical_abuse', 'ace_household_substance', 'ace_household_mental_illness']


The two counts above are what **this extract produced** — the `analytic_N` figures in Table I.

---
## D — Reporting the two cohorts

**Suppression applies to MCS only.** MCS is UKDS Safeguarded Tier 1a; YRBS is open CDC data and
is reported in full.

Some MCS-derived values are not reported following disclosure-control review. Additional values may be withheld to prevent recovery by subtraction or comparison across tables.

Every withheld value carries the same neutral marker and no reason, so a reader cannot tell
which rule applied to it. The operational detail — the rule, its threshold and which one fired —
is in `src/publication.py`, in the tests, and in the optional restricted file at the end of this
section. **Whether the threshold itself belongs in the paper's methods is a licence and
institutional question, not one this code can answer.**

In [13]:
# The disclosure rule lives in `publication`. Its operational detail — the threshold, and which
# rule withheld what — stays in source, in the tests and in the restricted private file below.
# Public output carries one neutral marker and no reason, so a reader cannot tell a small cell
# from one withheld to stop a subtraction.
SAVE_PRIVATE_COUNTS = False   # exact withheld values, for disclosure review — see section E

public_count = publication.public_count
breakdown = publication.breakdown
withheld_rows = publication.withheld_rows
NOT_REPORTED = publication.NOT_REPORTED

print("MCS-derived values may be withheld following disclosure-control review; "
      "YRBS is open CDC data and is reported in full")

MCS-derived values may be withheld following disclosure-control review; YRBS is open CDC data and is reported in full


### The sample flow, and what it commits the rest of the notebook to

Raw, excluded and analytic are an exact partition: any two give the third. So a small exclusion
is not protected by withholding it alone — and the analytic total is not this table's alone
either. It reappears as the denominator of the pillar table, as the total behind every
characteristic breakdown, and inside the missingness audit's own flow section. Withholding it
here and printing it there would withhold nothing at all, so this cell decides once and the
rest of the notebook follows.

In [14]:
flow, ANALYTIC_TOTAL_WITHHELD = [], {}
for cohort, raw, keep, pillars in (
        ("MCS", mcs_raw, mcs_analytic, mcs_shared_pillars),
        ("YRBS", yrbs_raw, yrbs_analytic, yrbs_shared_pillars)):
    raw_n = len(raw)
    analytic_n = int(keep.sum())
    excluded = raw_n - analytic_n
    all_five_missing = int(pillars.isna().all(axis=1).sum())

    if raw_n != analytic_n + excluded:
        raise ValueError(f"{cohort}: the sample flow does not partition as expected")

    # A withheld exclusion takes the analytic total with it — otherwise raw minus analytic
    # gives it straight back. Every later table reads this decision rather than taking its own.
    small = cohort == "MCS" and 0 < excluded < publication.SUPPRESS_BELOW
    ANALYTIC_TOTAL_WITHHELD[cohort] = small

    # NUMERIC COLUMNS STAY NUMERIC, and a withheld value is blank. A marker belongs in the
    # display, not mixed into a column a reader will parse as a number.
    flow.append(dict(
        cohort=cohort, raw_n=raw_n,
        excluded_missing_shared_pillar=None if small else excluded,
        analytic_n=None if small else analytic_n,
        missing_all_five_pillars=(None if cohort == "MCS"
                                  and 0 < all_five_missing < publication.SUPPRESS_BELOW
                                  else all_five_missing)))

cohort_flow = pd.DataFrame(flow)
display(publication.show(cohort_flow,
                         ["cohort", "raw_n", "excluded_missing_shared_pillar", "analytic_n",
                          "missing_all_five_pillars"]).fillna(NOT_REPORTED))

### Ethnicity did not survive as a single scheme

The two surveys do not encode ethnicity in the same way. The next step is to assess which
categories can be represented consistently *within* each cohort, rather than across them: MCS has
an Asian category and no Hispanic one, YRBS the reverse, and no crosswalk exists between them.
Every subgroup comparison later is therefore **within-cohort** — a Black–White gap in MCS and a
Black–White gap in YRBS are two findings, not one comparison.

**The collapse is about cell size, not comparability.** The conformal cells are sex × ethnicity,
and four coarse categories already leave MCS near the minimum cell size those cells need — a
previous authorised analysis recorded several of its calibration cells as short. The six native
`FDCE0600` categories would make twelve MCS cells instead of eight, splitting an already starved
cell and separating Indian from Pakistani-and-Bangladeshi, both close to the Tier 1a suppression
floor of ten.

| MCS `FDCE0600` (6-cat) | coarse | | YRBS `raceeth` (8-cat) | coarse |
|---|---|---|---|---|
| 1 White | White | | 5 White | White |
| 3 Indian | Asian | | 3 Black | Black |
| 4 Pakistani and Bangladeshi | Asian | | 6, 7 Hispanic | Hispanic |
| 5 Black or Black British | Black | | 1 AI/AN, 2 Asian, 4 NH/OPI, 8 multiple | Other |
| 2 Mixed | Mixed-or-Other | | | |
| 6 Other (inc Chinese) | Mixed-or-Other | | | |

The unreduced `attr_ethnicity` is persisted alongside it, because the collapse cannot be undone
and it is what retains the distinctions the coarse map removes.

**The YRBS source is chosen at run time.** The recoder prefers CDC `race4`, which follows the
standard coding, and falls back to `raceeth`. The two are not interchangeable: the `raceeth` map
in use was decoded by crosstabbing against the Q4 Hispanic item rather than taken from the
standard CDC ordering. The cell below reports which column was used, because that is the
assumption to re-check first if the input file is ever regenerated.

In [15]:
mcs_collapse, yrbs_collapse = features.COARSE_ATTRIBUTE_MAP["attr_ethnicity_coarse"]
mcs_attributes["attr_ethnicity_coarse"] = mcs_collapse(mcs_raw)
yrbs_attributes["attr_ethnicity_coarse"] = yrbs_collapse(yrbs_raw)

yrbs_source = "race4" if "race4" in yrbs_raw.columns else "raceeth"
print(f"YRBS coarse ethnicity built from `{yrbs_source}`"
      f"{'' if yrbs_source == 'race4' else '  <- fallback, decoded from the data'}")

# `Series.map` turns an uncovered code into NaN without complaint, so anyone classified in the
# fine attribute and not here fell through the collapse.
for label, attrs in (("MCS", mcs_attributes), ("YRBS", yrbs_attributes)):
    fine, coarse = attrs["attr_ethnicity"], attrs["attr_ethnicity_coarse"]
    if int((fine.notna() & coarse.isna()).sum()):
        raise ValueError(f"{label}: some respondents carry an ethnicity code the coarse map "
                         f"does not cover, and became NaN silently")
    if list(attrs.columns) != ATTRIBUTES + ["attr_ethnicity_coarse"]:
        raise ValueError(f"{label}: attribute columns are not the four base attributes plus "
                         f"the collapse")

# THE SAME CATEGORIES ARE BROKEN DOWN TWICE — here over the full cohort, and in section D over
# the analytic sample. Suppressing each on its own is not enough: a category of 100 in one and
# 95 in the other passes every single-cell rule and still says five of that category were
# excluded. So the two are decided together and the same set is hidden in both.
mcs_full_ethnicity = mcs_attributes["attr_ethnicity_coarse"].fillna("missing").value_counts()
mcs_analytic_ethnicity = (mcs_attributes.loc[mcs_analytic, "attr_ethnicity_coarse"]
                          .fillna("missing").value_counts())
ETHNICITY_WITHHELD = publication.paired_withholding(
    mcs_full_ethnicity, mcs_analytic_ethnicity, "MCS")

# BEFORE EITHER TABLE IS SHOWN. Where hiding a second category leaves only one non-zero
# hidden difference, subtraction recovers it and neither table may be published as it stands.
if not PUBLIC_NOTEBOOK and publication.UNRESOLVED in set(ETHNICITY_WITHHELD.values()):
    print("unresolved paired suppression — the categories and their counts in both tables:")
    _pair = pd.DataFrame({"full_cohort": mcs_full_ethnicity,
                          "analytic_sample": mcs_analytic_ethnicity}).fillna(0).astype(int)
    _pair["difference"] = _pair["full_cohort"] - _pair["analytic_sample"]
    display(_pair)
# UNCONDITIONAL. Two tables that give a withheld value back by subtraction are a disclosure
# problem inside the approved environment as much as outside it, so this stops the run
# whichever mode it is in. The detail is printed above, under the guard.
publication.require_resolved(ETHNICITY_WITHHELD, "coarse ethnicity, full cohort against "
                             "analytic sample")

for label, attrs in (("MCS", mcs_attributes), ("YRBS", yrbs_attributes)):
    coarse = attrs["attr_ethnicity_coarse"]
    hidden = tuple(ETHNICITY_WITHHELD) if label == "MCS" else ()
    print(f"\n{label}  {breakdown(coarse, label, len(coarse), also_withheld=hidden)}")

if ETHNICITY_WITHHELD:
    print(f"\n{len(ETHNICITY_WITHHELD)} ethnicity categor(ies) withheld from BOTH tables, so "
          f"neither the counts nor their difference is recoverable from the pair")

### Where the two cohorts differ, pillar by pillar

The composite prevalence gap is not one number. Decomposing it says which categories differ
between cohorts; the leave-one-pillar-out sensitivity in notebook 02 then says whether those
differences matter for transfer.

In [16]:
pillar_labels = {"ace_physical_abuse": "physical", "ace_sexual_abuse": "sexual",
                 "ace_emotional_abuse": "emotional",
                 "ace_household_mental_illness": "mental",
                 "ace_household_substance": "substance"}


def pillar_counts(pillars, analytic):
    """Numerator and denominator per measure, and the mean number of pillars per respondent.

    A pillar is 0/1, so its numerator is the column sum over the analytic sample. Keeping the
    numerator rather than only the proportion is what lets the display suppress a small cell.
    """
    kept = pillars.loc[analytic, list(data.SHARED_PILLARS)]
    reported = kept.sum(axis=1)
    n = len(kept)
    counts = {label: (int(kept[column].sum()), n) for column, label in pillar_labels.items()}
    counts["composite_ge1"] = (int((reported >= 1).sum()), n)
    counts["all5_positive"] = (int((reported == 5).sum()), n)
    return counts, float(reported.mean())


mcs_counts, mcs_mean_pillars = pillar_counts(mcs_pillars, mcs_analytic)
yrbs_counts, yrbs_mean_pillars = pillar_counts(yrbs_pillars, yrbs_analytic)

rows = []
for measure, (m_num, m_den) in mcs_counts.items():
    y_num, y_den = yrbs_counts[measure]
    m_prev, y_prev = m_num / m_den, y_num / y_den
    # A small MCS cell takes its prevalence and the cross-cohort ratio with it: either would
    # recover the count from the denominator printed beside it. BOTH TAILS — a prevalence of
    # 99.7% says three respondents are in the complementary cell.
    suppressed = publication.withhold_rate(m_num, m_den, "MCS")
    rows.append({
        "measure": measure,
        "mcs_n": None if suppressed else m_num,
        # The denominator IS the analytic total. Where the flow withheld it, it stays withheld
        # here, or the two tables together give it back.
        "mcs_denominator": None if ANALYTIC_TOTAL_WITHHELD["MCS"] else m_den,
        "mcs_prevalence": publication.public_rate(m_num, m_den, "MCS", places=4),
        "yrbs_n": y_num,
        "yrbs_denominator": y_den,
        "yrbs_prevalence": round(y_prev, 4),
        "ratio": None if (suppressed or not m_prev) else round(y_prev / m_prev, 3),
    })

pillar_prevalence = pd.DataFrame(rows)

# THE MEAN NUMBER OF PILLARS IS NOT PUBLISHED, and the reason is arithmetic rather than
# cautious: it equals the sum of the five pillar numerators over the denominator printed beside
# every row, so with one pillar withheld the mean gives that numerator straight back. It is kept
# for internal reading and for the private disclosure file.
mean_pillars_internal = {"MCS": mcs_mean_pillars, "YRBS": yrbs_mean_pillars}

display(publication.show(pillar_prevalence,
                         ["measure", "mcs_n", "mcs_denominator", "mcs_prevalence",
                          "yrbs_n", "yrbs_denominator", "yrbs_prevalence",
                          "ratio"]).fillna(NOT_REPORTED))
if not PUBLIC_NOTEBOOK:
    print(f"mean pillars per respondent (internal only): "
          f"MCS {mcs_mean_pillars:.4f}, YRBS {yrbs_mean_pillars:.4f}")

Seven count-based rows — the five pillars plus `composite_ge1` and `all5_positive`. The mean number of pillars per respondent is not published: it reverses a withheld numerator against the denominator printed beside every row. The figures are what **this extract produced**.

---
## E — Missingness and cohort characteristics

The strict rule costs the YRBS side heavily, and why matters for how the loss is read: the
adverse-experience module was **optional** in 2023, so most excluded respondents are whole schools
where it was never fielded. A previous authorised analysis recorded those respondents as
answering non-module items at very nearly the full rate, which is the basis for treating the
absence as administrative rather than as a refusal — a sampling fact rather than informative
missingness. The output below is what this extract gives.

Tier 1a: the audit suppresses MCS cells below 10 and rounds them to the nearest 10.

In [17]:
missingness = data.missingness_audit(
    mcs_features, yrbs_features,
    mcs_pillars=mcs_pillars, yrbs_pillars=yrbs_pillars,
    mcs_attributes=mcs_attributes, yrbs_attributes=yrbs_attributes,
    mcs_raw_n=len(mcs_raw), yrbs_raw_n=len(yrbs_raw))
inputs.save_table(missingness, "missingness_audit.csv")

print(missingness["section"].value_counts().to_dict())

`missingness_audit.csv` is a **derived diagnostic**, not a canonical modelling input: it is the §III sample-flow evidence, and notebook 02 does not read it. Passing no attribute table skips section 4 with a recorded row saying so, never silently. The section counts are what **this extract produced**.

### Who the YRBS outcome rule leaves out

The strict rule costs the YRBS side heavily, and the audit above says how much. It does not say
whether the respondents it removes look like the ones it keeps. The table below sets the two
groups side by side on the demographics the cohort already carries.

**Excluded means one thing.** The harmonised outcome could not be defined for that respondent —
at least one of the five shared pillars is unobserved. It is not a refusal, a data error or a
quality judgement, and it is not read as one here.

**The optional module may account for much of it.** The 2023 adverse-experience items were not
fielded everywhere, so a site that omitted them contributes excluded respondents in bulk. That
is a plausible route to the missingness and it is consistent with what the audit shows.

**The data cannot say why a site did not administer a module.** Nothing in this extract records
a school's, district's or state's reasons, and no row below is evidence about one. A difference
between the two groups is a description of who is present, not an explanation of who decided
what.

**Differences here bear on representativeness.** Where the included and excluded groups differ,
the analytic sample is not a miniature of the surveyed population, and every YRBS figure in this
work describes the sample that could be given an outcome. The comparison is descriptive
throughout: sizes and directions, **no p-values**, and no test of any null hypothesis.

In [18]:
# The live mask from section C, passed rather than re-derived: eligibility is defined once.
# Group sizes are denominators inside the function and do not come out of it.
yrbs_inclusion = data.yrbs_inclusion_comparison(yrbs_attributes, yrbs_analytic)
display(publication.show(yrbs_inclusion, list(data.INCLUSION_COLUMNS)))

print("difference = included - excluded: years for the mean age, percentage points for a "
      "category, unitless for the standardised difference")

### Exact counts, for disclosure review only

`SAVE_PRIVATE_COUNTS` writes every value this notebook withheld, with its exact count, its
denominator, its percentage and the rule that withheld it, to the restricted MCS location — never
into the repository, and never displayed here. It is off by default.

That file exists because a reviewer has to see what was withheld in order to approve what was
shown. Notebook 01 records no person-level row in it: every entry is a cell count or a rate.

In [19]:
characteristics = []
for cohort, attributes, keep in (("MCS", mcs_attributes, mcs_analytic),
                                 ("YRBS", yrbs_attributes, yrbs_analytic)):
    analytic = attributes.loc[keep]
    total, age = len(analytic), analytic["attr_age"]
    sex = analytic["attr_sex"].map({0.0: "male", 1.0: "female"})
    hidden = tuple(ETHNICITY_WITHHELD) if cohort == "MCS" else ()

    # THE BREAKDOWN COUNTS SUM TO THE ANALYTIC TOTAL, so where the flow withheld that total
    # this table reports shares alone. Otherwise the two together hand it back.
    withhold_total = ANALYTIC_TOTAL_WITHHELD[cohort]

    # AGE MIN AND MAX ARE NOT PUBLISHED. They describe the two most extreme respondents in the
    # cohort, which is a different kind of statement from a mean.
    characteristics.append(dict(
        cohort=cohort,
        analytic_n=None if withhold_total else total,
        age_mean=round(float(age.mean()), 2), age_sd=round(float(age.std()), 2),
        sex=breakdown(sex, cohort, total, share_only=withhold_total),
        ethnicity=breakdown(analytic["attr_ethnicity_coarse"], cohort, total,
                            also_withheld=hidden, share_only=withhold_total)))

sample_characteristics = pd.DataFrame(characteristics)
display(publication.show(sample_characteristics,
                         ["cohort", "analytic_n", "age_mean", "age_sd", "sex",
                          "ethnicity"]).fillna(NOT_REPORTED))

if any(ANALYTIC_TOTAL_WITHHELD.values()):
    print("\nshares only where the analytic total is not reported. The raw total is public, so "
          "\nthe denominator is bounded rather than hidden — flag this pair for review.")

The file carries the measure, the category, the exact count, the
denominator, the percentage and the rule that withheld it. **None of that appears in public
output**, where one neutral marker covers every reason.

In [20]:
if SAVE_PRIVATE_COUNTS:
    withheld = withheld_rows(mcs_attributes["attr_ethnicity_coarse"], "MCS",
                             "ethnicity_coarse_full_cohort",
                             also_withheld=tuple(ETHNICITY_WITHHELD))
    # The cross-table decision itself, so a reviewer can see WHY a category was hidden in both.
    for _category, _why in ETHNICITY_WITHHELD.items():
        withheld.append(dict(measure="ethnicity_coarse_cross_table", category=str(_category),
                             count=int(mcs_full_ethnicity.get(_category, 0)),
                             denominator=int(mcs_analytic_ethnicity.get(_category, 0)),
                             percent=None, reason=_why))
    mcs_analytic_attrs = mcs_attributes.loc[mcs_analytic]
    for field in ("attr_sex", "attr_ethnicity_coarse"):
        withheld += withheld_rows(mcs_analytic_attrs[field], "MCS", f"{field}_analytic_sample",
                                  also_withheld=tuple(ETHNICITY_WITHHELD))

    def record_scalar(measure, category, count, denominator):
        """A standalone figure, withheld under the primary rule alone."""
        if 0 < count < publication.SUPPRESS_BELOW:
            withheld.append(dict(measure=measure, category=category, count=int(count),
                                 denominator=int(denominator), reason="primary",
                                 percent=round(100 * count / denominator, 4)
                                         if denominator else None))

    record_scalar("sample_flow", "excluded_missing_shared_pillar",
                  (~mcs_analytic).sum(), len(mcs_analytic))
    record_scalar("sample_flow", "missing_all_five_shared_pillars",
                  mcs_shared_pillars.isna().all(axis=1).sum(), len(mcs_shared_pillars))
    for measure, (numerator, denominator) in mcs_counts.items():
        record_scalar("pillar_prevalence", measure, numerator, denominator)

    # The mean number of pillars is withheld from public output because it reverses a pillar
    # count, so it belongs in this file rather than nowhere.
    for cohort, value in mean_pillars_internal.items():
        if cohort == "MCS":
            withheld.append(dict(measure="pillar_prevalence", category="mean_n_pillars",
                                 count=None, denominator=None, percent=None,
                                 reason="reverses_a_withheld_numerator", value=value))

    exact_counts = pd.DataFrame(withheld)
    PRIVATE_COUNTS_PATH = config.MCS_FEATURES.parent / "notebook_01_exact_counts.csv"
    PRIVATE_COUNTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    exact_counts.to_csv(PRIVATE_COUNTS_PATH, index=False)   # replaced, never appended
    print("withheld MCS values written to the restricted MCS location")
else:
    print("SAVE_PRIVATE_COUNTS is False — no private file written, no directory created.")

### Do the predictors relate to the outcome in the same way in each cohort?

Transfer from MCS to YRBS loses discrimination. That is measured in notebook 02; it does not say
what about the two cohorts causes it. One observable thing that could sit behind such a loss is
whether the predictors themselves relate to the outcome differently on the two sides, and this
section asks that and nothing more. It belongs here because it is a property of the harmonised
frames, not of any fitted model.

**The estimand.** Each retained predictor is related to the `>=2` outcome on its own, separately
in each cohort — one one-predictor logistic fit per predictor, the predictor standardised within
its own cohort. The nominal predictor enters as contrasts against its reference level rather than
as a slope through its codes, so no ordering is asserted that its levels do not have.

**Only an aggregate is shown.** The table below counts *terms*: how many were constructed, how
many both cohorts could answer, how many point the same way, and how far the two sets of
estimates track each other. **No predictor is named, no coefficient is shown, and no largest
difference or reversal is listed.** Nothing here is a count of respondents, a denominator or a
category frequency.

**What it cannot do.** It does not decompose the transfer gap — nothing is refitted with a
difference removed. It does not identify a cause: country, age range, instrument and sampling
frame differ together. **Agreement does not establish measurement invariance or measurement
equivalence.** Two cohorts can reach the same coefficient from different constructs, and this
design tests nothing about whether a question means the same thing on both sides. Because each
predictor is standardised within its own cohort, a difference between coefficients can reflect
both a different relationship with the outcome and a different spread of the predictor.

**MCS-derived, and pending disclosure review.** Every figure below is computed partly from MCS
Sweep 6 records. The candidate file this writes is for review inside the approved environment;
it is not cleared for release. The summary is displayed only when this notebook runs with
`PUBLIC_NOTEBOOK = False`; in public mode the computation and the candidate are unchanged and no
value is shown.

In [21]:
# The four frames are the ones section C built and this notebook still holds; nothing is
# re-read and no second outcome definition is introduced. The library restricts each cohort
# with the canonical analytic mask and composes the outcome through `make_outcome`.
ASSOCIATION_THRESHOLD = 2

# The eleven columns the candidate carries. `n_not_estimable` and
# `n_not_reportable_for_release` are deliberately absent: the second is a disclosure fact about
# withheld cells and does not belong in a table offered for review as a scientific result.
ASSOCIATION_SUMMARY_COLUMNS = [
    "threshold", "n_harmonised_predictors", "n_association_terms", "n_estimable_terms",
    "n_direction_terms", "n_same_direction", "n_reversed_direction", "sign_agreement",
    "n_correlated_terms", "pearson_r", "spearman_rho",
]

# THE INTERNAL ROUTE, BY NAME. It keeps the estimates the small-count rule will not release, so
# the term counts describe what the data answered rather than what may be shown. Only the
# aggregate below leaves this cell; the term-level frame is not displayed.
association_terms = data.compare_cohort_associations_internal(
    mcs_features, yrbs_features,
    mcs_pillars=mcs_pillars, yrbs_pillars=yrbs_pillars,
    threshold=ASSOCIATION_THRESHOLD)

# `n_harmonised_predictors` is read from the registry, never typed. The term count differs from
# it because the nominal predictor contributes one contrast per non-reference level.
cohort_association_summary = (
    data.association_summary_internal(association_terms)
    .rename(columns={"n_attempted_terms": "n_association_terms"})
    .assign(threshold=f">={ASSOCIATION_THRESHOLD}",
            n_harmonised_predictors=len(data.FEATURE_COLUMNS))
    .loc[:, ASSOCIATION_SUMMARY_COLUMNS])

# A CANDIDATE, NOT AN APPROVED OUTPUT. It is always computed and always written, so the review
# has something to read; it goes to the working root, never into the repository.
publication.save_table(
    cohort_association_summary, "cohort_association_summary.csv",
    ASSOCIATION_SUMMARY_COLUMNS)

# THE DISPLAY IS GUARDED. This cell is not in Notebook 01's retain_output_cells list,
# so its stored internal output is removed from a generated public copy. The guard also
# suppresses its values if the notebook is deliberately executed in public mode.
# Public mode prints a statement and no number.
if not PUBLIC_NOTEBOOK:
    print(f"{len(data.FEATURE_COLUMNS)} harmonised predictors -> "
          f"{int(cohort_association_summary['n_association_terms'].iloc[0])} association "
          f"terms at >={ASSOCIATION_THRESHOLD}")
    display(publication.show(cohort_association_summary, ASSOCIATION_SUMMARY_COLUMNS))
    print("MCS-derived. Disclosure review is required before any of it is used or released.")
else:
    print("cohort-association summary computed and written as a review candidate; "
          "MCS-derived and awaiting disclosure review, so no value is shown here")

---
## F — Canonical outputs for modelling

Six artefacts: the harmonised features, the ACE pillars and the evaluation-only attributes, per
cohort. Notebooks 02 and 04 consume these and never re-read a raw cohort. MCS frames are
person-level and resolve under `$MCS_DATA_DIR`, outside the clone; YRBS is open CDC data and goes
under the working root. `config` refuses a root that resolves inside the repository, at a home
directory or at a filesystem root, with symlinks resolved first.

**Not written: the outcome, and the splits.** The outcome is a deterministic collapse of the
pillars through `make_outcome`, which is its one definition — a stored copy could disagree with
it. Splits hold rows, and notebook 02 reconstructs them from features, outcome and the seed.

**Persisted at raw scale.** Nothing here is standardised. Cohort standardisation — the reported
`lineage='cs'` — is fitted **per split** inside `build_splits`, so there is no single standardised
cohort to store, and doing it here would fit the scaler across training and test rows together.
Because each frame is standardised against itself the target transform is transductive; notebook
02 owns that decision and reports it.

These six are deterministic recodes of the raw cohorts, so every run rebuilds all of them and the
write replaces what it finds. Nothing here is a result.

**The two feature frames carry a version stamp.** They are written through
`data.write_harmonised_features`, which puts a `.provenance.json` sidecar beside each parquet
recording `data.PREPROCESSING_VERSION` and the harmonised schema. Notebooks 02 and 04 read them
through `data.read_harmonised_features`, which refuses a frame built under a different version —
so a change to a recoder cannot reach a model through a parquet left over from an earlier run.
The pillars and attributes carry no sidecar, because neither is built from the feature recoders
or the model feature schema; a change that alters *their* construction needs a stamp of its own.

**How the write behaves.** The four pillar and attribute artefacts are prepared beside their
destinations first, and only when all four have been written does anything replace a
destination; a failure during preparation removes the temporary files without touching a
destination. The two feature frames are written afterwards, each atomically, so a parquet is
never left beside a sidecar naming a version it was not built under. Each individual replacement
is atomic, so none can be left half-written. **The collection is not a transaction**: if a
replacement fails partway, the destinations already replaced stay replaced and the notebook
raises rather than undoing them.

In [22]:
# Destinations named rather than derived: reaching an MCS constant is what trips the read gate,
# so building this list is where the cell stops without MCS access.
#
# The two harmonised feature frames go through `data.write_harmonised_features`, which records
# the preprocessing version and the schema beside each parquet so notebooks 02 and 04 cannot
# read a frame recoded by an earlier version. The pillars and attributes carry no sidecar --
# see the note beside `data.FEATURE_PROVENANCE_SUFFIX`.
feature_artefacts = [
    (mcs_features,    config.MCS_FEATURES),
    (yrbs_features,   config.YRBS_FEATURES),
]
other_artefacts = [
    (mcs_pillars,     config.MCS_PILLARS),
    (yrbs_pillars,    config.YRBS_PILLARS),
    (mcs_attributes,  config.MCS_ATTRIBUTES),
    (yrbs_attributes, config.YRBS_ATTRIBUTES),
]
artefacts = feature_artefacts + other_artefacts

import os

prepared = []
try:
    for frame, path in other_artefacts:
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(path.suffix + ".partial")
        try:
            frame.to_parquet(tmp, index=True)
        except Exception:
            tmp.unlink(missing_ok=True)   # the one being written when it failed
            raise
        prepared.append((tmp, path, frame))

    for tmp, path, frame in prepared:
        os.replace(tmp, path)             # os.replace is atomic; .to_parquet is not

    # Written last, and each atomically, so no feature parquet is ever left beside a sidecar
    # naming a version it was not built under.
    for frame, path in feature_artefacts:
        data.write_harmonised_features(frame, path)

    # Names only. A row count here is the cohort size, and a path is the restricted location.
    print(f"{len(artefacts)} canonical artefacts written to their configured locations")
except Exception:
    # Whatever is left unreplaced. A tmp already renamed is gone, so missing_ok covers it.
    for tmp, _, _ in prepared:
        tmp.unlink(missing_ok=True)
    raise

### Checking the six footers

**This check runs when you execute the notebook**; it has not been run against the real artefacts. It reads each file's Parquet footer — the recorded schema and row count — and compares them with the frame just written. No respondent data is loaded, so it confirms that the footer and schema are as expected and **not** that every data page in the file reads back.

In [23]:
import pyarrow.parquet as pq

# Written with index=True, so each footer should record the frame's columns in order followed
# by the persisted index — and which field that is comes from the footer's own pandas metadata
# rather than from assuming whatever trails the columns is an index.
for frame, path in artefacts:
    meta = pq.ParquetFile(path)
    on_disk, expected = list(meta.schema_arrow.names), list(frame.columns)
    index_columns = list((meta.schema_arrow.pandas_metadata or {}).get("index_columns", []))
    if on_disk[:len(expected)] != expected:
        raise ValueError(f"{path.name}: columns differ from the frame written, or are reordered")
    if len(index_columns) != 1 or not isinstance(index_columns[0], str):
        raise ValueError(f"{path.name}: expected one physical index column, the footer records "
                         f"{index_columns}")
    if on_disk[len(expected):] != index_columns:
        raise ValueError(f"{path.name}: the footer names an index the schema does not trail")
    if meta.metadata.num_rows != len(frame):
        raise ValueError(f"{path.name}: the footer records a different number of rows from the "
                         f"frame written")

# Names only: a row count here is the cohort size.
print(f"{len(artefacts)} footers validated against the frames written")

---

**Next:** `02_models_and_transfer.ipynb` composes the outcome from these pillars, builds the
splits, and fits the source models.

---
## The tables this notebook offers the manuscript

Five candidates, each built from a named list of columns rather than by removing columns from a
working frame. They go to `publication_candidates/` below the configured working root, never
into the repository. The YRBS inclusion comparison is the one that carries no MCS-derived
figure; it is a candidate on the same terms as the rest.

**Writing them is not clearance.** The MCS-derived figures need my review and, where the
agreement requires it, institutional and UK Data Service approval before they are committed.

In [24]:
publication.save_table(
    cohort_flow, "cohort_flow.csv",
    ["cohort", "raw_n", "excluded_missing_shared_pillar", "analytic_n",
     "missing_all_five_pillars"])

publication.save_table(
    sample_characteristics, "sample_characteristics.csv",
    ["cohort", "analytic_n", "age_mean", "age_sd", "sex", "ethnicity"])

publication.save_table(
    pillar_prevalence, "outcome_prevalence.csv",
    ["measure", "mcs_n", "mcs_denominator", "mcs_prevalence",
     "yrbs_n", "yrbs_denominator", "yrbs_prevalence", "ratio"])

# YRBS only, and no count reaches it: means, percentages, and the differences between them.
publication.save_table(
    yrbs_inclusion, "yrbs_inclusion_comparison.csv",
    ["variable", "level", "included_value", "excluded_value", "difference", "measure"])

# THE MISSINGNESS AUDIT IS RATES, AND A RATE ON A KNOWN DENOMINATOR IS A COUNT.
# `data.MISSINGNESS_COLUMNS` is a fixed schema, so the columns are named rather than guessed.
#
# Two of the six sections carry counts the other tables already suppress — `sample_flow` is the
# same raw-to-analytic partition as cohort_flow.csv, and `complete_case` is an exact `n` per
# cohort — so both stay out. Two more stay out because their DENOMINATORS are subgroup and
# outcome-arm sizes this notebook does not hold, and a rate whose denominator I cannot name is
# a rate I cannot check.
PUBLIC_MISSINGNESS_SECTIONS = ("feature_cohort", "pillar_cohort")
missingness_public = missingness[
    missingness["section"].isin(PUBLIC_MISSINGNESS_SECTIONS)].copy()

# EACH SECTION IS MEASURED OVER ITS OWN POPULATION, and a rate only gives its count back
# against the population it was measured on. `feature_cohort` is per-feature missingness
# within the analytic sample; `pillar_cohort` is per-pillar missingness across the whole
# pillar frame, before the inclusion rule removes anyone. Reading a pillar rate against the
# analytic total names a different cell from the one the rate describes.
MCS_MISSINGNESS_DENOMINATORS = {"feature_cohort": int(mcs_analytic.sum()),
                                "pillar_cohort": len(mcs_pillars)}

unnamed = sorted(set(missingness_public["section"]) - set(MCS_MISSINGNESS_DENOMINATORS))
if unnamed:
    raise ValueError(f"missingness section(s) {unnamed} reach the public table with no stated "
                     f"population, so the cell behind their rates cannot be checked against "
                     f"the small-count rule. Name the population each was measured over, or "
                     f"leave the section out of the public table.")

# The denominators decide what is withheld and go no further. A denominator printed beside a
# published rate is the count the rule has just withheld.
_denominators = missingness_public["section"].map(MCS_MISSINGNESS_DENOMINATORS)
_withhold = [publication.withhold_rate(publication.implied_numerator(rate, n), n, "MCS")
             for rate, n in zip(missingness_public["pct_missing"], _denominators)]

# `diff_pp` is the MCS rate minus the YRBS one, and the YRBS rate is published beside it — so
# a withheld MCS rate takes the difference with it, or the subtraction gives it straight back.
# `flag` becomes a nullable Boolean first: a plain Boolean column has no missing value to put
# there, and one that accepts a blank only because it happens to hold mixed types is one
# refactor away from raising.
missingness_public["flag"] = missingness_public["flag"].astype("boolean")
missingness_public.loc[_withhold, ["pct_missing", "diff_pp"]] = pd.NA
missingness_public.loc[_withhold, "flag"] = pd.NA

missingness_public["pct_missing"] = missingness_public["pct_missing"].round(2)
missingness_public["compare_pct"] = missingness_public["compare_pct"].round(2)
missingness_public["diff_pp"] = missingness_public["diff_pp"].round(2)

publication.save_table(
    missingness_public, "missingness_summary.csv",
    ["section", "cohort", "key", "pct_missing", "compare_pct", "diff_pp", "flag"])

if any(_withhold):
    print(f"  {sum(_withhold)} missingness rate(s) not reported")


---
## Before any of this is committed

1. run the tests;
2. run this notebook, then 02, then 03 — stopping if 03's reconciliation fails — then 04;
3. run `python scripts/check_public_outputs.py`;
4. review every MCS-derived table and figure by eye;
5. obtain any institutional or UK Data Service approval the agreement requires;
6. commit the executed public notebooks and the reviewed publication outputs.

Step 3 reports known patterns. It is not clearance, and steps 4 and 5 are not optional.